In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

In [2]:
%pip install -qqq --upgrade ultralytics pyyaml transformers opencv-python-headless seaborn timm "numpy==1.26.4" "scikit-learn==1.3.2" huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/

## Setup Kaggle

In [3]:
import json, cv2
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from collections import Counter
from transformers import ViTForImageClassification
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
import numpy as np
from PIL import Image, ImageOps
from datetime import datetime, timedelta
from glob import glob
import os, shutil, yaml, re, unicodedata
from collections import defaultdict
import matplotlib.pyplot as plt
from torch.amp import autocast
from timm.data.mixup import Mixup
from tqdm.auto import tqdm
from tqdm.notebook import tqdm
#  INPUT datasets
DATASETS = [
    "/kaggle/input/bracol-for-yolov8-detection/BRACOL_REVIEWED_ANNOTATIONS/BRACOL_REVIEWED",
    "/kaggle/input/coffee-leaves/BRACOL-VALIDADO.v11i.yolov11",
    "/kaggle/input/coffee-leaves/Coffee Leaf Diseases - OD.v10i.yolov11",
    "/kaggle/input/coffee-leaves/Coffee Leaf.v7-train-valid-test-70-20-10.yolov11",
    "/kaggle/input/coffee-leaves/Coffee leaf diseases classification.v5i.yolov11",
    "/kaggle/input/coffee-leaves/Coffee_Disease_BRACOL.v3i.yolov11",
    "/kaggle/input/coffee-leaves/Folhas.v8i.yolov11",
    "/kaggle/input/coffee-leaves/Leaf plants clasification.v13i.yolov11",
    "/kaggle/input/coffee-leaves/My First Project.v23i.yolov11",
    "/kaggle/input/coffee-leaves/My First Project.v2i.yolov11",
    "/kaggle/input/coffee-leaves/NeuralCoffe.v15i.yolov11",
    "/kaggle/input/coffee-leaves/cashew",
    "/kaggle/input/coffee-leaves/coffee leaf disease.v3i.yolov11",
    "/kaggle/input/coffee-leaves/coffee leaf.v7i.yolov11",
    "/kaggle/input/coffee-leaves/coffee leaves.v1i.yolov11",
    "/kaggle/input/coffee-leaves/coffee.v2i.yolov11",
    "/kaggle/input/coffee-leaves/datasets",
    "/kaggle/input/coffee-leaves/nhan_dien_benh_tren_la_ca_phe.v2i.yolov11"
]

#  Canonical classes
CANONICAL_NAMES = [
    "nam_ri_sat", 
    "sau_duc_la", 
    "khoe_manh", 
    "phoma", 
    "than_thu",
]
SYNONYMS = {
    "nam_ri_sat": {"rust", "coffee_rust"}, #"nam ri sat", "nam_ri_sat", "leaf rust", "leaf_rust", "Rust"
    "sau_duc_la": {"miner", "Miner", "leaf_miner", "leafminor", "leaf minor", "leaf_minor", "Mineiro", "mineiro", "coffee_miner"}, #
    "khoe_manh": {"healthy", "Healthy_Leaf", "healthy_leaf", "coffee_healthy", "Healthy-Leaf"}, #"Healthy", 
    "phoma": {"phoma", "Phoma", "coffee_phoma"}, #
    "than_thu": {"than thu", "than_thu", "anthracnose", "Anthracnose", "Antracnose"},
}

#  Paths 
MERGED_ROOT = "/kaggle/working/merged"
ROI_ROOT    = "/kaggle/working/roi"
CKPT_ROOT   = "/kaggle/working/checkpoints"
#  YOLO train settings 
YOLO_MODEL = "yolo11s.pt"
IMGSZ_YOLO = 640
EPOCHS_YOLO = 20
DEVICE_YOLO = 'cuda'
WORKERS_YOLO = os.cpu_count() // 2
#  ViT settings 
VIT_MODEL_NAME = 'google/vit-base-patch16-224'
ROI_SIZE = 224
EPOCHS_VIT = 60
BASE_LR_VIT = 1e-5
BATCH_VIT = 32
MIXUP = 0.5
LABEL_SMOOTH = 0.1
WEIGHT_DECAY = 0.05
def strip_accents(s: str) -> str:
    s = unicodedata.normalize("NFD", s)
    return "".join(ch for ch in s if not unicodedata.combining(ch))

def norm_name(name: str) -> str:
    n = strip_accents(name.strip())
    n = re.sub(r"[^a-zA-Z0-9\s\-]+", "_", n).strip('_')
    print(n)
    return n

def canonicalize(name: str):
    n = norm_name(name)
    for cano, pool in SYNONYMS.items():
        if n in pool or n == cano:
            return cano
    return None
print(datetime.utcnow() + timedelta(hours=7))

E0000 00:00:1767064202.874187      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767064202.927419      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

2025-12-30 10:10:18.561494


## Cấu hình dataset, hợp nhất các bộ datasets và lọc ROI

In [4]:

for split in ["train", "valid", "test"]:
    Path(f"{MERGED_ROOT}/{split}/images").mkdir(parents=True, exist_ok=True)
    Path(f"{MERGED_ROOT}/{split}/labels").mkdir(parents=True, exist_ok=True)

SPLIT_ALIASES = {
    "train": ["train"],
    "valid":  ["test"],
    "test": ["valid", "val"]
}

def resolve_split_dirs(ds_root: Path):
    #Trả về dict: split_chuan ('train'/'valid'/'test') -> (img_dir, lbl_dir)
    #Hỗ trợ 2 layout:
    #  A) Roboflow:    <root>/<split>/images, <root>/<split>/labels
    #  B) Ultralytics: <root>/images/<split>, <root>/labels/<split>
    
    results = {}
    # --- Layout: <root>/<alias>/{images,labels} ---
    for split_std, aliases in SPLIT_ALIASES.items():
        for alias in aliases:
            img_a = ds_root / alias / "images"
            lbl_a = ds_root / alias / "labels"
            if img_a.exists() and lbl_a.exists():
                results[split_std] = (img_a, lbl_a)
                break
    return results

stats = {s: defaultdict(int) for s in ["train","valid","test"]}

for ds in DATASETS:
    ds_root = Path(ds)
    if not ds_root.exists():
        print(f"[WARN] Dataset không tồn tại: {ds}")
        continue
    yaml_path = ds_root / "data.yaml"
    names = None
    if yaml_path.exists():
        with open(yaml_path,'r') as f:
            meta = yaml.safe_load(f)
        names = meta.get('names')
        if isinstance(names, dict):
            try:
                names = [names[k] for k in sorted(names.keys(), key=lambda x: int(x))]
            except Exception:
                # fallback: sắp theo khóa như chuỗi
                names = [names[k] for k in sorted(names.keys())]
    id_map = {}
    if names is not None:
        for old_id, name in enumerate(names):
            cano = canonicalize(str(name))
            if cano is None:
                print(f"[WARN] '{name}' không map -> bỏ bbox lớp này.")
                continue
            id_map[old_id] = CANONICAL_NAMES.index(cano)
    prefix = ds_root.name.replace(' ','_')    
    # Phát hiện cấu trúc split và lặp
    split_dirs = resolve_split_dirs(ds_root)

    for split_std, (img_dir, lbl_dir) in split_dirs.items():
    
        for txt in lbl_dir.glob('*.txt'):
            lines_out = []
    
            with open(txt,'r') as f:
                for ln in f:
                    ln = ln.strip()
                    if not ln:
                        continue
                    parts = ln.split()
                    old_id = int(parts[0])
                    if old_id not in id_map:
                        stats[split_std]['labels_dropped'] += 1
                        continue
                    parts[0] = str(id_map[old_id])
                    lines_out.append(' '.join(parts))
    
            # CHỈ GIỮ ẢNH NẾU CÓ LABEL HỢP LỆ
            if not lines_out:
                continue
    
            img_name = txt.stem + '.jpg'   # fallback
            for ext in ['.jpg','.jpeg','.png','.bmp','.webp']:
                cand = img_dir / (txt.stem + ext)
                if cand.exists():
                    img_name = cand.name
                    break
    
            src_img = img_dir / img_name
            if not src_img.exists():
                continue
    
            # copy image
            shutil.copy2(
                src_img,
                Path(f"{MERGED_ROOT}/{split_std}/images") / f"{prefix}_{img_name}"
            )
    
            # write label
            out_lbl = Path(f"{MERGED_ROOT}/{split_std}/labels") / f"{prefix}_{txt.name}"
            with open(out_lbl,'w') as g:
                g.write('\n'.join(lines_out)+'\n')
    
            stats[split_std]['images'] += 1
            stats[split_std]['labels_kept'] += 1

merged_yaml = {
    'path': ROI_ROOT,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    #'nc': {len(CANONICAL_NAMES)},
    'names': {i:n for i,n in enumerate(CANONICAL_NAMES)}
}
with open(Path("/kaggle/working")/'merged_data.yaml','w',encoding='utf-8') as f:
    yaml.safe_dump(merged_yaml, f, sort_keys=False, allow_unicode=True)
print('MERGE DONE')
for s in ["train","valid","test"]:
    print(s, dict(stats[s]))

print(datetime.utcnow() + timedelta(hours=7))

Cercospora
[WARN] 'Cercospora' không map -> bỏ bbox lớp này.
Miner
Phoma
Rust
[WARN] 'Rust' không map -> bỏ bbox lớp này.
Cercospora
[WARN] 'Cercospora' không map -> bỏ bbox lớp này.
Miner
Phoma
Rust
[WARN] 'Rust' không map -> bỏ bbox lớp này.
abiotic disorder
[WARN] 'abiotic disorder' không map -> bỏ bbox lớp này.
cercospora
[WARN] 'cercospora' không map -> bỏ bbox lớp này.
healthy
rust
sooty mold
[WARN] 'sooty mold' không map -> bỏ bbox lớp này.
healthy
miner
phoma
rust
cerscospora
[WARN] 'cerscospora' không map -> bỏ bbox lớp này.
healthy
miner
phoma
rust
Cercosporiose
[WARN] 'Cercosporiose' không map -> bỏ bbox lớp này.
Ferrugem
[WARN] 'Ferrugem' không map -> bỏ bbox lớp này.
Mineiro
Phoma
cercosporiose
[WARN] 'cercosporiose' không map -> bỏ bbox lớp này.
healthy
miner
phoma
rust
Algal
[WARN] 'Algal' không map -> bỏ bbox lớp này.
Antracnose
Bacterial-Leaf-Blight
[WARN] 'Bacterial-Leaf-Blight' không map -> bỏ bbox lớp này.
Bacterial-Spot
[WARN] 'Bacterial-Spot' không map -> bỏ bbox 

In [5]:

for split in ['train','valid','test']:
    imgs = list(Path(f"{MERGED_ROOT}/{split}/images").iterdir())
    lbls = list(Path(f"{MERGED_ROOT}/{split}/labels").iterdir())
    print(split, "images:", len(imgs), "labels:", len(lbls))
# Tạo cấu trúc ROI_ROOT
for split in ['train','valid','test']:
    Path(f"{ROI_ROOT}/{split}/images").mkdir(parents=True, exist_ok=True)
    Path(f"{ROI_ROOT}/{split}/labels").mkdir(parents=True, exist_ok=True)

roi_stats = {s: defaultdict(int) for s in ['train','valid','test']}

for split in ['train','valid','test']:
    src_images = Path(f"{MERGED_ROOT}/{split}/images")
    src_labels = Path(f"{MERGED_ROOT}/{split}/labels")

    dst_images = Path(f"{ROI_ROOT}/{split}/images")
    dst_labels = Path(f"{ROI_ROOT}/{split}/labels")

    for img_path in src_images.iterdir():
        if not img_path.is_file():
            continue
        if img_path.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp','.webp'}:
            continue

        label_path = src_labels / (img_path.stem + '.txt')
        if not label_path.exists():
            roi_stats[split]['no_label'] += 1
            continue

        # kiểm tra label có class hợp lệ không
        valid = False
        with open(label_path, 'r') as f:
            for ln in f:
                parts = ln.strip().split()
                if len(parts) < 5:
                    continue
                cid = int(parts[0])
                if 0 <= cid < len(CANONICAL_NAMES):
                    valid = True
                    break

        if not valid:
            roi_stats[split]['no_valid_class'] += 1
            continue

        shutil.copy2(img_path, dst_images / img_path.name)
        shutil.copy2(label_path, dst_labels / label_path.name)
        roi_stats[split]['kept'] += 1

for s in ['train','valid','test']:
    print(f"[ROI {s}]", dict(roi_stats[s]))

#src_path = "/kaggle/input/coffee-leaves-disease-5-classes-merged-dataset/roi"
#dst_path = "/kaggle/working/roi"
#shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
#yaml_path = "/kaggle/input/coffee-leaves-disease-5-classes-merged-dataset/merged_data.yaml"
#shutil.copy2(yaml_path, "/kaggle/working")
def count_per_class(root):
    counts = {c: 0 for c in CANONICAL_NAMES}

    label_dir = Path(root) / "labels"
    if not label_dir.exists():
        return counts

    for txt in label_dir.glob("*.txt"):
        present = set()
        with open(txt, "r") as f:
            for ln in f:
                parts = ln.strip().split()
                if len(parts) < 5:
                    continue
                cid = int(parts[0])
                if 0 <= cid < len(CANONICAL_NAMES):
                    present.add(CANONICAL_NAMES[cid])

        # mỗi class chỉ +1 / ảnh
        for c in present:
            counts[c] += 1

    return counts

print("ROI train counts:", count_per_class(f"{ROI_ROOT}/train"))
print("ROI valid counts:", count_per_class(f"{ROI_ROOT}/valid"))
print("ROI test counts:",  count_per_class(f"{ROI_ROOT}/test"))

print(datetime.utcnow() + timedelta(hours=7))

if os.path.exists(MERGED_ROOT):
    shutil.rmtree(MERGED_ROOT)
    print(f"Đã xóa thư mục{MERGED_ROOT}")

train images: 16767 labels: 16767
valid images: 3059 labels: 3059
test images: 2398 labels: 2398
[ROI train] {'kept': 16767}
[ROI valid] {'kept': 3059}
[ROI test] {'kept': 2398}
ROI train counts: {'nam_ri_sat': 2058, 'sau_duc_la': 2377, 'khoe_manh': 2397, 'phoma': 4231, 'than_thu': 5952}
ROI valid counts: {'nam_ri_sat': 796, 'sau_duc_la': 944, 'khoe_manh': 353, 'phoma': 867, 'than_thu': 277}
ROI test counts: {'nam_ri_sat': 429, 'sau_duc_la': 491, 'khoe_manh': 372, 'phoma': 567, 'than_thu': 599}
2025-12-30 10:17:35.677709
Đã xóa thư mục/kaggle/working/merged


## Train YOLO

In [6]:
"""
best = None
cand = glob('kaggle/working/runs/detect/*/weights/best.pt')
if cand:
    best = sorted(cand)[-1]
model_yolo = YOLO(YOLO_MODEL)
print('Using YOLO weights:', best or YOLO_MODEL)

results = model_yolo.train(
    data=str(Path("/kaggle/working") / 'merged_data.yaml'),
    imgsz=IMGSZ_YOLO,
    epochs=EPOCHS_YOLO,
    batch=-1,
    device=DEVICE_YOLO,
    amp=True,
    workers=WORKERS_YOLO,
    patience=10,
    cache='ram'
)
print('Best weights:', model_yolo.trainer.best)
"""

'\nbest = None\ncand = glob(\'kaggle/working/runs/detect/*/weights/best.pt\')\nif cand:\n    best = sorted(cand)[-1]\nmodel_yolo = YOLO(YOLO_MODEL)\nprint(\'Using YOLO weights:\', best or YOLO_MODEL)\n\nresults = model_yolo.train(\n    data=str(Path("/kaggle/working") / \'merged_data.yaml\'),\n    imgsz=IMGSZ_YOLO,\n    epochs=EPOCHS_YOLO,\n    batch=-1,\n    device=DEVICE_YOLO,\n    amp=True,\n    workers=WORKERS_YOLO,\n    patience=10,\n    cache=\'ram\'\n)\nprint(\'Best weights:\', model_yolo.trainer.best)\n'

## Transforms theo ImageNet

In [7]:
"""
IMAGENET_MEAN=(0.485,0.456,0.406); IMAGENET_STD=(0.229,0.224,0.225)
try:
    rand_aug=[transforms.RandAugment(num_ops=2,magnitude=9)]
except Exception:
    rand_aug=[transforms.ColorJitter(0.2,0.2,0.2,0.1)]
    
class ROIDatasetYOLO(Dataset):
    def __init__(self, root, img_dir="images", lbl_dir="labels", transform=None):
        self.root = Path(root)
        self.img_dir = self.root / img_dir
        self.lbl_dir = self.root / lbl_dir
        self.transform = transform

        self.items = []
        for img_path in self.img_dir.glob('*'):
            if img_path.suffix.lower() not in {'.jpg','.jpeg','.png'}:
                continue
            lbl_path = self.lbl_dir / (img_path.stem + '.txt')
            if lbl_path.exists():
                self.items.append((img_path, lbl_path))

        self._ph = Image.new('RGB',(ROI_SIZE,ROI_SIZE),(0,0,0))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_path, lbl_path = self.items[idx]

        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = self._ph.copy()

        # đọc class từ YOLO label (lấy class đầu tiên)
        with open(lbl_path, 'r') as f:
            line = f.readline().strip().split()
            y = int(line[0])

        if self.transform:
            try:
                img = self.transform(img)
            except:
                img = self.transform(self._ph.copy())

        return img, y

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(ROI_SIZE, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(0.5),
    *rand_aug,
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

eval_tfms = transforms.Compose([
    transforms.Resize(int(ROI_SIZE*1.15)),
    transforms.CenterCrop(ROI_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
train_ds = ROIDatasetYOLO(f"{ROI_ROOT}/train", transform=train_tfms)
val_ds   = ROIDatasetYOLO(f"{ROI_ROOT}/valid", transform=eval_tfms)
test_ds  = ROIDatasetYOLO(f"{ROI_ROOT}/test",  transform=eval_tfms)

print("Số ROI train/val/test:", len(train_ds), len(val_ds), len(test_ds))
labels_train = [y for _, y in train_ds]
cnt = Counter(labels_train)

class_freq = np.array([cnt.get(i,0) for i in range(len(CANONICAL_NAMES))], dtype=np.float64)
class_freq[class_freq==0] = 1
class_weight = 1.0 / class_freq

sample_weights = np.array([class_weight[y] for y in labels_train], dtype=np.float64)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

mixup_fn = Mixup(mixup_alpha=MIXUP, cutmix_alpha=0.2, prob=1.0 if True else 0.0, switch_prob=0.0, mode='batch', label_smoothing=LABEL_SMOOTH, num_classes=len(CANONICAL_NAMES))

def collate_fn(batch):
    imgs,labels=list(zip(*batch))
    imgs=torch.stack(imgs,dim=0); labels=torch.tensor(labels,dtype=torch.long)
    imgs,labels=mixup_fn(imgs,labels)
    return imgs,labels

NUM_WORKERS = WORKERS_YOLO
PIN_MEMORY=torch.cuda.is_available()

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_VIT,
    sampler=sampler,
    num_workers=WORKERS_YOLO,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_VIT,
    shuffle=False,
    num_workers=WORKERS_YOLO,
    pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_ds, 
    batch_size=BATCH_VIT,
    shuffle=False, 
    num_workers=WORKERS_YOLO,
    pin_memory=torch.cuda.is_available()
)

print(datetime.utcnow() + timedelta(hours=7))
"""

'\nIMAGENET_MEAN=(0.485,0.456,0.406); IMAGENET_STD=(0.229,0.224,0.225)\ntry:\n    rand_aug=[transforms.RandAugment(num_ops=2,magnitude=9)]\nexcept Exception:\n    rand_aug=[transforms.ColorJitter(0.2,0.2,0.2,0.1)]\n    \nclass ROIDatasetYOLO(Dataset):\n    def __init__(self, root, img_dir="images", lbl_dir="labels", transform=None):\n        self.root = Path(root)\n        self.img_dir = self.root / img_dir\n        self.lbl_dir = self.root / lbl_dir\n        self.transform = transform\n\n        self.items = []\n        for img_path in self.img_dir.glob(\'*\'):\n            if img_path.suffix.lower() not in {\'.jpg\',\'.jpeg\',\'.png\'}:\n                continue\n            lbl_path = self.lbl_dir / (img_path.stem + \'.txt\')\n            if lbl_path.exists():\n                self.items.append((img_path, lbl_path))\n\n        self._ph = Image.new(\'RGB\',(ROI_SIZE,ROI_SIZE),(0,0,0))\n\n    def __len__(self):\n        return len(self.items)\n\n    def __getitem__(self, idx):\n      

## Train

In [8]:
"""
device='cuda' if torch.cuda.is_available() else 'cpu'
model=ViTForImageClassification.from_pretrained(VIT_MODEL_NAME)
num_ftrs=model.classifier.in_features; model.classifier=nn.Linear(num_ftrs,len(CANONICAL_NAMES))
model.to(device)

criterion=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
optimizer=optim.AdamW(model.parameters(), lr=BASE_LR_VIT, weight_decay=WEIGHT_DECAY)
scheduler=optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1,EPOCHS_VIT))
scaler=torch.amp.GradScaler(device, enabled=torch.cuda.is_available())

def accuracy_from_logits(logits, targets):
    preds=logits.argmax(dim=1); y=targets if targets.ndim==1 else targets.argmax(dim=1)
    return (preds==y).float().mean().item()

history={'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
best_val=-1.0
best_dir=Path(CKPT_ROOT)/'vit'
best_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(1,EPOCHS_VIT+1):
    print(datetime.utcnow() + timedelta(hours=7))
    model.train(); tot_l=tot_a=n=0
    for imgs,labels in train_loader:
        imgs=imgs.to(device); labels=labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device, enabled=torch.cuda.is_available()):
            outputs=model(imgs); logits=outputs.logits if hasattr(outputs,'logits') else outputs
            loss=criterion(logits, labels if labels.ndim==1 else labels.argmax(dim=1))
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        acc=accuracy_from_logits(logits.detach(), labels.detach())
        tot_l+=loss.item(); tot_a+=acc; n+=1
    tr_loss, tr_acc = tot_l/max(1,n), tot_a/max(1,n)

    model.eval(); tot_l=tot_a=n=0
    with torch.no_grad():
        for imgs,labels in val_loader:
            imgs=imgs.to(device); labels=labels.to(device)
            with autocast(device, enabled=torch.cuda.is_available()):
                outputs=model(imgs); logits=outputs.logits if hasattr(outputs,'logits') else outputs
                loss=criterion(logits, labels)
            acc=accuracy_from_logits(logits, labels)
            tot_l+=loss.item(); tot_a+=acc; n+=1
    va_loss, va_acc = tot_l/max(1,n), tot_a/max(1,n)
    scheduler.step()
    history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss); history['val_acc'].append(va_acc)
    print(f"Epoch {epoch}/{EPOCHS_VIT} | train_loss={tr_loss:.4f} acc={tr_acc:.4f} | val_loss={va_loss:.4f} acc={va_acc:.4f}")
    if va_acc>best_val:
        best_val=va_acc
        model.config.id2label={i:c for i,c in enumerate(CANONICAL_NAMES)}
        model.config.label2id={c:i for i,c in enumerate(CANONICAL_NAMES)}
        model.save_pretrained(best_dir, safe_serialization=True)
        with open(best_dir/'metrics.json','w') as f: json.dump({'val_acc': float(best_val)}, f, indent=2)
        print('** Saved best ViT to', best_dir)
print(datetime.utcnow() + timedelta(hours=7))
"""

'\ndevice=\'cuda\' if torch.cuda.is_available() else \'cpu\'\nmodel=ViTForImageClassification.from_pretrained(VIT_MODEL_NAME)\nnum_ftrs=model.classifier.in_features; model.classifier=nn.Linear(num_ftrs,len(CANONICAL_NAMES))\nmodel.to(device)\n\ncriterion=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)\noptimizer=optim.AdamW(model.parameters(), lr=BASE_LR_VIT, weight_decay=WEIGHT_DECAY)\nscheduler=optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1,EPOCHS_VIT))\nscaler=torch.amp.GradScaler(device, enabled=torch.cuda.is_available())\n\ndef accuracy_from_logits(logits, targets):\n    preds=logits.argmax(dim=1); y=targets if targets.ndim==1 else targets.argmax(dim=1)\n    return (preds==y).float().mean().item()\n\nhistory={\'train_loss\':[], \'train_acc\':[], \'val_loss\':[], \'val_acc\':[]}\nbest_val=-1.0\nbest_dir=Path(CKPT_ROOT)/\'vit\'\nbest_dir.mkdir(parents=True, exist_ok=True)\n\nfor epoch in range(1,EPOCHS_VIT+1):\n    print(datetime.utcnow() + timedelta(hours=7))\n    mo

## Evaluation

In [9]:
"""
# plots
fig=plt.figure(figsize=(12,5))
ax1=fig.add_subplot(1,2,1)
ax1.plot(range(1,len(history['train_loss'])+1), history['train_loss'], label='Train Loss')
ax1.plot(range(1,len(history['val_loss'])+1),   history['val_loss'],   label='Val Loss')
ax1.legend(); ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax2=fig.add_subplot(1,2,2)
ax2.plot(range(1,len(history['train_acc'])+1), history['train_acc'], label='Train Acc')
ax2.plot(range(1,len(history['val_acc'])+1),   history['val_acc'],   label='Val Acc')
ax2.legend(); ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Acc')
fig.tight_layout(); fig.savefig(Path(CKPT_ROOT)/'vit_loss_acc.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)

# Test
print("\n Đánh giá trên Test ")
model=ViTForImageClassification.from_pretrained(best_dir).to(device).eval()
y_true=[]; y_pred=[]
with torch.no_grad():
    for imgs,labels in test_loader:
        imgs=imgs.to(device); outputs=model(imgs)
        logits=outputs.logits if hasattr(outputs,'logits') else outputs
        preds=logits.argmax(dim=1).cpu().numpy().tolist()
        y_pred+=preds; y_true+=labels.numpy().tolist()

labels_present=sorted(set(y_true)|set(y_pred))
names_present=[CANONICAL_NAMES[i] for i in labels_present]

rep=classification_report(y_true,y_pred,labels=labels_present,target_names=names_present,digits=4,zero_division=0)
print(rep)
with open(Path(CKPT_ROOT)/'vit_classification_report.txt','w',encoding='utf-8') as f: f.write(rep)
cm=confusion_matrix(y_true,y_pred,labels=labels_present)
fig,ax=plt.subplots(figsize=(max(8,0.6*len(names_present)), max(6,0.5*len(names_present))))
#sns.heatmap(cm[:, ::-1], annot=True, fmt='d', cmap='Blues', xticklabels=CANONICAL_NAMES[::-1], yticklabels=CANONICAL_NAMES, ax=ax)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CANONICAL_NAMES, yticklabels=CANONICAL_NAMES, ax=ax)
ax.set_title('Ma trận nhầm lẫn'); ax.set_xlabel('Dự đoán'); ax.set_ylabel('Thực tế')
fig.tight_layout(); fig.savefig(Path(CKPT_ROOT)/'vit_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
# load models
yolo_det = YOLO(YOLO_MODEL if not list(Path('kaggle/working/runs/detect').glob('*/weights/best.pt')) else sorted(Path('kaggle/working/runs/detect').glob('*/weights/best.pt'))[-1].as_posix())
vit_cls  = ViTForImageClassification.from_pretrained(Path(CKPT_ROOT)/'vit').to(device).eval()

vit_tf = transforms.Compose([
    transforms.Resize(ROI_SIZE),
    transforms.CenterCrop(ROI_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))
])

def predict_yolo_vit(img_path:str, conf=0.25, iou=0.45):
    img = Image.open(img_path).convert('RGB')
    W,H = img.size
    res = yolo_det.predict(source=img_path, conf=conf, iou=iou, verbose=False)
    outputs=[]
    if len(res)>0 and res[0].boxes is not None and len(res[0].boxes)>0:
        boxes = res[0].boxes.xyxy.cpu().numpy()
        for bb in boxes:
            x1,y1,x2,y2 = map(int, bb)
            roi = img.crop((x1,y1,x2,y2)).resize((ROI_SIZE,ROI_SIZE))
            t = vit_tf(roi).unsqueeze(0).to(vit_cls.device)
            with torch.no_grad():
                logits = vit_cls(t).logits
                pred = int(logits.argmax(dim=1).item())
                outputs.append({
                    'bbox':[x1,y1,x2,y2],
                    'class_id': pred,
                    'class_name': CANONICAL_NAMES[pred]
                })
    return outputs

#if os.path.exists(ROI_ROOT):
#    shutil.rmtree(ROI_ROOT)
#    print(f"Đã xóa thư mục{ROI_ROOT}")
print(datetime.utcnow() + timedelta(hours=7))
"""

'\n# plots\nfig=plt.figure(figsize=(12,5))\nax1=fig.add_subplot(1,2,1)\nax1.plot(range(1,len(history[\'train_loss\'])+1), history[\'train_loss\'], label=\'Train Loss\')\nax1.plot(range(1,len(history[\'val_loss\'])+1),   history[\'val_loss\'],   label=\'Val Loss\')\nax1.legend(); ax1.set_title(\'Loss\'); ax1.set_xlabel(\'Epoch\'); ax1.set_ylabel(\'Loss\')\nax2=fig.add_subplot(1,2,2)\nax2.plot(range(1,len(history[\'train_acc\'])+1), history[\'train_acc\'], label=\'Train Acc\')\nax2.plot(range(1,len(history[\'val_acc\'])+1),   history[\'val_acc\'],   label=\'Val Acc\')\nax2.legend(); ax2.set_title(\'Accuracy\'); ax2.set_xlabel(\'Epoch\'); ax2.set_ylabel(\'Acc\')\nfig.tight_layout(); fig.savefig(Path(CKPT_ROOT)/\'vit_loss_acc.png\', dpi=200, bbox_inches=\'tight\')\nplt.show()\nplt.close(fig)\n\n# Test\nprint("\n Đánh giá trên Test ")\nmodel=ViTForImageClassification.from_pretrained(best_dir).to(device).eval()\ny_true=[]; y_pred=[]\nwith torch.no_grad():\n    for imgs,labels in test_loader:

## Test với những ảnh ngoài dataset

In [10]:
"""
if os.path.exists('/kaggle'):
    TESTDIR = '/kaggle/input/test-images'
    CKPT_ROOT   = "/kaggle/working/checkpoints"
else:
    # Fallback for local execution if TESTDIR is not set
    TESTDIR = 'test_images'
    CKPT_ROOT = Path("./checkpoints")
    print(f"Thư mục test ngoài dataset không được cấu hình, sẽ dùng {TESTDIR}")
TESTDIR = Path(TESTDIR)
print(TESTDIR)
def safe_open_image(path):
    try:
        return Image.open(path).convert('RGB')
    except Exception:
        img_cv = cv2.imread(str(path))
        if img_cv is None:
            raise ValueError(f'Không thể đọc ảnh: {path}')
        img_cv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
        return Image.fromarray(img_cv)

test_tfms = transforms.Compose([
    transforms.Resize(int(ROI_SIZE * 1.15)),
    transforms.CenterCrop(ROI_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
if not TESTDIR.exists():
    print(f"Thư mục test ngoài dataset không tồn tại tại {TESTDIR}")
else:
    all_files = [p for p in TESTDIR.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]

    num_samples = len(all_files)

    if num_samples > 0:
        # ================= LOAD MODELS =================
        yolo = YOLO("/kaggle/working/runs/detech/train/weights/best.pt")
        vit  = ViTForImageClassification.from_pretrained(best_dir).to(device)
        vit.eval()
        
        # ================= COLLECT IMAGES =================
        all_files = [p for p in TESTDIR.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png'}]
        if not all_files:
            print("Không có ảnh test.")
            raise SystemExit
        
        # ================= INFERENCE =================
        predictions = []
        
        for img_path in all_files:
            img = safe_open_image(img_path)
            img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
        
            dets = yolo(img_np, verbose=False)[0]
        
            results = []
            for box in dets.boxes:
                x1,y1,x2,y2 = map(int, box.xyxy[0])
                roi = img.crop((x1,y1,x2,y2))
        
                roi_tensor = test_tfms(roi).unsqueeze(0).to(device)
                with torch.no_grad():
                    logits = vit(roi_tensor).logits
                    probs = torch.softmax(logits, dim=1)[0]
        
                cls_id = probs.argmax().item()
                cls_name = vit.config.id2label[cls_id]
                conf = probs[cls_id].item()
        
                results.append({
                    "bbox": [x1,y1,x2,y2],
                    "class": cls_name,
                    "confidence": conf
                })
        
            predictions.append({
                "image": str(img_path),
                "detections": results
            })
        
        # ================= VISUALIZE =================
        max_show = min(20, len(predictions))
        cols = 5
        rows = (max_show + cols - 1) // cols
        
        fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4))
        axes = axes.flatten()
        
        for i in range(max_show):
            img = safe_open_image(predictions[i]["image"])
            axes[i].imshow(img)
        
            for det in predictions[i]["detections"]:
                x1,y1,x2,y2 = det["bbox"]
                cls = det["class"]
                conf = det["confidence"]
        
                axes[i].add_patch(
                    plt.Rectangle((x1,y1), x2-x1, y2-y1,
                                  fill=False, edgecolor='red', linewidth=2)
                )
                axes[i].text(x1, y1-4, f"{cls} {conf:.2f}",
                             color='red', fontsize=8)
        
            axes[i].axis('off')
        
        for j in range(i+1, len(axes)):
            axes[j].axis('off')
        
        plt.tight_layout()
        fig.savefig(CKPT_ROOT / "external_yolo_vit_test.png", dpi=200)
        plt.show()
        plt.close()
        
        # ================= SAVE JSON =================
        with open(CKPT_ROOT / "external_yolo_vit_test.json", "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
    else:
        print(f"Không tìm thấy ảnh nào để test trong thư mục {TESTDIR}.")
"""
print(datetime.utcnow() + timedelta(hours=7))


2025-12-30 10:17:37.609142
